### New experiments using OSM. 
* We will pull out the relevant data in tags of osm
* Then we will filter to Exeter

In [12]:
import pyrosm
import geopandas as gpd
import pandas as pd

PBF_FILE = r"osm_data\devon-260326.osm.pbf"

print("Reading OSM data...")
osm = pyrosm.OSM(PBF_FILE)

# Helper function to safely save a layer
def save_layer(gdf, name):
    if gdf is not None and len(gdf) > 0:
        print(f"{name}: {len(gdf):,} features")
        gdf.to_file(f"devon_{name}.gpkg", driver="GPKG")
        print(f"✓ Saved devon_{name}.gpkg")
    else:
        print(f"{name}: empty or not found")

# ── Buildings ─────────────────────────────────────────────
print("\nReading buildings...")
buildings = osm.get_buildings()
save_layer(buildings, "buildings")

# ── Network / Roads ───────────────────────────────────────
print("\nReading roads/network...")
network = osm.get_network(network_type="all")
save_layer(network, "network")

# ── Landuse ───────────────────────────────────────────────
print("\nReading landuse...")
landuse = osm.get_landuse()
save_layer(landuse, "landuse")

# ── POIs ──────────────────────────────────────────────────
print("\nReading POIs...")
pois = osm.get_pois()
save_layer(pois, "pois")

# ── Natural features ──────────────────────────────────────
print("\nReading natural features...")
natural = osm.get_natural()
save_layer(natural, "natural")

# ── Waterways ─────────────────────────────────────────────
print("\nReading waterways...")
waterways = osm.get_data_by_custom_criteria(custom_filter={"waterway": True})
save_layer(waterways, "waterways")

# ── Boundaries ────────────────────────────────────────────
print("\nReading boundaries...")
boundaries = osm.get_data_by_custom_criteria(custom_filter={"boundary": True})
save_layer(boundaries, "boundaries")

print("\nDone!")

Reading OSM data...

Reading buildings...
buildings: 370,964 features
✓ Saved devon_buildings.gpkg

Reading roads/network...


c:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\osvenv\Lib\site-packages\pyrosm\networks.py:37: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  edges, nodes = prepare_geodataframe(


network: 203,552 features
✓ Saved devon_network.gpkg

Reading landuse...
landuse: 52,427 features
✓ Saved devon_landuse.gpkg

Reading POIs...
pois: 43,855 features
✓ Saved devon_pois.gpkg

Reading natural features...
natural: 64,592 features
✓ Saved devon_natural.gpkg

Reading waterways...
waterways: 17,831 features
✓ Saved devon_waterways.gpkg

Reading boundaries...
boundaries: 658 features
✓ Saved devon_boundaries.gpkg

Done!


### Post filtering of OSM data 
* Removing irrelevant columns
* Finding filter columns

#### Buildings

In [10]:
import geopandas as gpd
gdf_polygon = gpd.read_file(r"osm_data\devon_buildings.gpkg")
gdf_polygon.columns

Index(['addr:city', 'addr:country', 'addr:housenumber', 'addr:housename',
       'addr:postcode', 'addr:place', 'addr:street', 'email', 'name',
       'opening_hours', 'operator', 'phone', 'ref', 'url', 'visible',
       'website', 'building', 'amenity', 'building:flats', 'building:levels',
       'building:material', 'building:min_level', 'building:use', 'craft',
       'height', 'internet_access', 'landuse', 'levels', 'office', 'shop',
       'source', 'start_date', 'wikipedia', 'id', 'timestamp', 'version',
       'tags', 'osm_type', 'changeset', 'geometry'],
      dtype='object')

In [28]:
recommended_columns = ['addr:city', 'addr:country', 'addr:housenumber', 'addr:housename',
       'addr:postcode', 'addr:place', 'addr:street','name',
       'opening_hours', 'website', 'building', 'amenity', 'building:flats', 'building:levels',
       'building:material', 'building:min_level', 'building:use','craft',
       'height', 'internet_access', 'landuse', 'levels', 'office', 'shop',
       'source','geometry']

In [ ]:
gdf_polygon.building.isna().any()


np.False_

In [34]:
invalid = gdf_polygon[~gdf_polygon.is_valid]
print(len(invalid))

45


In [43]:
gdf_polygon.building.unique()

array(['train_station', 'yes', 'commercial', 'market', 'retail',
       'parking', 'office', 'supermarket', 'university', 'apartments',
       'industrial', 'school', 'church', 'hut', 'cathedral', 'public',
       'hospital', 'hotel', 'residential', 'house', 'government',
       'farm_auxiliary', 'warehouse', 'dormitory', 'garages', 'detached',
       'terrace', 'boathouse', 'depot', 'semidetached_house', 'roof',
       'kindergarten', 'college', 'farm', 'fire_station', 'shed',
       'service', 'hall', 'lych_gate', 'transportation', 'hangar',
       'chapel', 'castle', 'civic', 'garage', 'barn', 'sports_centre',
       'Home Lodge', 'pavilion', 'pub', 'bunker', 'clubhouse',
       'beach_hut', 'toilets', 'bungalow', 'ruins', 'no', 'greenhouse',
       'yes;carport', 'bus_station', 'collapsed', 'glasshouse', 'stable',
       'manufacture', 'guardhouse', 'art_centre', 'historic', 'carport',
       'bandstand', 'grandstand', 'stadium', 'yurt', 'static_caravan',
       'substation', 'cabi

#### Landuse

In [7]:
gdf_landuse = gpd.read_file(r"osm_data\devon_landuse.gpkg")
gdf_landuse.columns

Index(['visible', 'tags', 'lon', 'timestamp', 'lat', 'version', 'id',
       'changeset', 'industrial', 'landuse', 'military', 'osm_type', 'basin',
       'construction', 'depot', 'farmland', 'farmyard', 'grass', 'meadow',
       'residential', 'greenfield', 'geometry'],
      dtype='object')

In [35]:
invalid = gdf_landuse[~gdf_landuse.is_valid]
print(len(invalid))

11


In [8]:
gdf_landuse.crs

<Projected CRS: EPSG:27700>
Name: OSGB36 / British National Grid
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: United Kingdom (UK) - offshore to boundary of UKCS within 49°45'N to 61°N and 9°W to 2°E; onshore Great Britain (England, Wales and Scotland). Isle of Man onshore.
- bounds: (-9.01, 49.75, 2.01, 61.01)
Coordinate Operation:
- name: British National Grid
- method: Transverse Mercator
Datum: Ordnance Survey of Great Britain 1936
- Ellipsoid: Airy 1830
- Prime Meridian: Greenwich

In [40]:
gdf_landuse.head()

,visible,tags,lon,timestamp,lat,version,id,changeset,industrial,landuse,...,basin,construction,depot,farmland,farmyard,grass,meadow,residential,greenfield,geometry
0,False,"{""disused"":""yes"",""name"":""Eylesbarrow Tin Mine""...",-3.977914,1654503709,50.496590,4,330109394,0.0,None,industrial,...,None,None,None,None,None,None,None,None,None,POINT (-3.97791 50.49659)
1,False,"{""name"":""Grenade Range""}",-3.347928,1255637686,50.679985,1,532323358,0.0,None,military,...,None,None,None,None,None,None,None,None,None,POINT (-3.34793 50.67999)
2,False,"{""name"":""Park Farm"",""note:retag"":""CS #48756035...",-2.993883,1692607471,50.751118,6,581133967,0.0,None,farm,...,None,None,None,None,None,None,None,None,None,POINT (-2.99388 50.75112)
3,False,"{""name"":""Highridge Farm"",""note:retag"":""CS #487...",-3.897673,1496238966,50.804760,4,800909134,0.0,None,farm,...,None,None,None,None,None,None,None,None,None,POINT (-3.89767 50.80476)
4,False,"{""name"":""North Tawton Boar Stud"",""note:retag"":...",-3.879628,1496238966,50.807281,3,800909353,0.0,None,farm,...,None,None,None,None,None,None,None,None,None,POINT (-3.87963 50.80728)


In [42]:
gdf_landuse.landuse.unique()

array(['industrial', 'military', 'farm', 'cemetery', 'retail', 'quarry',
       'forest', 'allotments', 'farmyard', 'commercial', 'village_green',
       'recreation_ground', 'grass', 'construction', 'farmland',
       'residential', 'ornamental', 'brownfield', 'landfill', 'orchard',
       'religious', 'railway', 'meadow', 'vineyard', 'depot',
       'churchyard', 'education', 'garages', 'paddock', 'plant_nursery',
       'greenhouse_horticulture', 'private', 'scout_camp', 'highway',
       'aquaculture', 'greenfield', 'civic', 'basin', 'flowerbed',
       'conservation', 'animal_keeping', 'observatory', 'utility',
       'healthcare', 'harbour', 'scrub', 'apiary', 'fishfarm',
       'grassnote'], dtype=object)

In [ ]:
gdf_landuse.landuse.isna().any()

np.False_

#### Natural

In [9]:
gdf_natural = gpd.read_file(r"osm_data\devon_natural.gpkg")
gdf_natural.columns

Index(['visible', 'tags', 'lon', 'timestamp', 'lat', 'version', 'id',
       'changeset', 'natural', 'tree', 'water', 'osm_type', 'cliff',
       'grassland', 'heath', 'peak', 'scrub', 'wetland', 'geometry'],
      dtype='object')

In [36]:
invalid = gdf_natural[~gdf_natural.is_valid]
print(len(invalid))

15


In [10]:
gdf_natural.crs

<Projected CRS: EPSG:27700>
Name: OSGB36 / British National Grid
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: United Kingdom (UK) - offshore to boundary of UKCS within 49°45'N to 61°N and 9°W to 2°E; onshore Great Britain (England, Wales and Scotland). Isle of Man onshore.
- bounds: (-9.01, 49.75, 2.01, 61.01)
Coordinate Operation:
- name: British National Grid
- method: Transverse Mercator
Datum: Ordnance Survey of Great Britain 1936
- Ellipsoid: Airy 1830
- Prime Meridian: Greenwich

In [49]:
gdf_natural.natural.unique()

array(['peak', 'cape', 'tree', 'bay', 'valley', 'headland', 'spring',
       'beach', 'cave_entrance', 'rocks', 'bare_rock', 'rock', 'stone',
       'cliff', 'wood', 'tree_stump', 'water', 'shrub', 'coastline',
       'wetland', 'mud', 'heath', 'grassland', 'scrub', 'sand', 'shingle',
       'sand_dunes', 'tree_row', 'scree', 'boulders', 'moor', 'grass',
       'cave', 'garden', 'earth_bank', 'gully', 'meadow', 'shrubbery',
       'tree_group', 'island', 'peninsula', 'wood;scrub'], dtype=object)

#### POI

In [12]:
gdf_pois = gpd.read_file(r"osm_data\devon_pois.gpkg")
gdf_pois.columns

Index(['visible', 'tags', 'lon', 'timestamp', 'lat', 'version', 'id',
       'changeset', 'addr:city', 'addr:country', 'addr:full',
       'addr:housenumber', 'addr:housename', 'addr:postcode', 'addr:place',
       'addr:street', 'email', 'name', 'opening_hours', 'operator', 'phone',
       'ref', 'url', 'website', 'amenity', 'atm', 'bank', 'bicycle_parking',
       'bicycle_rental', 'bar', 'building', 'building:levels', 'cafe',
       'college', 'drinking_water', 'fast_food', 'fountain', 'fuel',
       'gambling', 'internet_access', 'landuse', 'office', 'parking',
       'post_office', 'school', 'social_facility', 'source', 'start_date',
       'wikipedia', 'alcohol', 'appliance', 'bicycle', 'charity', 'clothes',
       'confectionery', 'craft', 'farm', 'fireplace', 'furniture',
       'garden_centre', 'gift', 'hairdresser', 'motorcycle', 'music',
       'organic', 'outdoor', 'religion', 'second_hand', 'shoes', 'shop',
       'tattoo', 'trade', 'wholesale', 'attraction', 'camp_site',


In [37]:
invalid = gdf_pois[~gdf_pois.is_valid]
print(len(invalid))

4


In [13]:
gdf_pois.crs

<Projected CRS: EPSG:27700>
Name: OSGB36 / British National Grid
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: United Kingdom (UK) - offshore to boundary of UKCS within 49°45'N to 61°N and 9°W to 2°E; onshore Great Britain (England, Wales and Scotland). Isle of Man onshore.
- bounds: (-9.01, 49.75, 2.01, 61.01)
Coordinate Operation:
- name: British National Grid
- method: Transverse Mercator
Datum: Ordnance Survey of Great Britain 1936
- Ellipsoid: Airy 1830
- Prime Meridian: Greenwich

#### waterways

In [14]:
gdf_waterways = gpd.read_file(r"osm_data\devon_waterways.gpkg")
gdf_waterways.columns

Index(['visible', 'tags', 'lon', 'timestamp', 'lat', 'version', 'id',
       'changeset', 'waterfall', 'waterway', 'osm_type', 'canal', 'dock',
       'fish_pass', 'geometry'],
      dtype='object')

In [38]:
invalid = gdf_waterways[~gdf_waterways.is_valid]
print(len(invalid))

5


In [16]:
gdf_waterways.crs

<Projected CRS: EPSG:27700>
Name: OSGB36 / British National Grid
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: United Kingdom (UK) - offshore to boundary of UKCS within 49°45'N to 61°N and 9°W to 2°E; onshore Great Britain (England, Wales and Scotland). Isle of Man onshore.
- bounds: (-9.01, 49.75, 2.01, 61.01)
Coordinate Operation:
- name: British National Grid
- method: Transverse Mercator
Datum: Ordnance Survey of Great Britain 1936
- Ellipsoid: Airy 1830
- Prime Meridian: Greenwich

In [56]:
gdf_waterways.waterway.unique()

array(['weir', 'lock_gate', 'rapids', 'turning_point', 'dock', 'boatyard',
       'dam', 'waterfall', 'stream_end', 'floodgate', 'flow_control',
       'canal', 'fuel', 'milestone', 'ladder', 'sluice_gate', 'stream',
       'river', 'drain', 'ditch', 'derelict_canal', 'landing_stage',
       'flowline', 'leat', 'fish_pass', 'tidal_channel', 'link'],
      dtype=object)

### Boundaries

In [17]:
gdf_boundaries = gpd.read_file(r"osm_data\devon_boundaries.gpkg")
gdf_boundaries.columns

Index(['visible', 'tags', 'lon', 'timestamp', 'lat', 'version', 'id',
       'changeset', 'name', 'operator', 'boundary', 'marker', 'start_date',
       'osm_type', 'website', 'admin_level', 'geometry'],
      dtype='object')

In [39]:
invalid = gdf_boundaries[~gdf_boundaries.is_valid]
print(len(invalid))

0


In [18]:
gdf_boundaries.crs

<Projected CRS: EPSG:27700>
Name: OSGB36 / British National Grid
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: United Kingdom (UK) - offshore to boundary of UKCS within 49°45'N to 61°N and 9°W to 2°E; onshore Great Britain (England, Wales and Scotland). Isle of Man onshore.
- bounds: (-9.01, 49.75, 2.01, 61.01)
Coordinate Operation:
- name: British National Grid
- method: Transverse Mercator
Datum: Ordnance Survey of Great Britain 1936
- Ellipsoid: Airy 1830
- Prime Meridian: Greenwich

In [59]:
gdf_boundaries.name.unique()

array([None, 'Stone Post (Boundary Marker 1885)', 'Ladywell Cross',
       "St John's Primary School nature reserve",
       'Goosemoor Nature Reserve', 'Braunton Burrows', 'Sand Dunes',
       'Ash Moor', 'Cranbrook Country Park',
       'East of Start Point Marine Conservation Zone', 'Topsham Road',
       'Cornwall', 'Devon', 'Somerset', 'Dorset', 'Exmoor National Park',
       'South West England', 'Torbay', 'Plymouth', 'Taunton Deane',
       'Dartmoor National Park', 'West Somerset', 'South Somerset',
       'West Dorset', 'East Devon', 'Exwick Ward', 'South Hams',
       'Teignbridge', 'Exeter', 'Cowick Ward', 'Duryard Ward',
       'Pennsylvania Ward', 'Mincinglake Ward', 'Pinhoe Ward',
       'Whipton & Barton Ward', 'St. Loyes Ward', 'Polsloe Ward',
       'Saint James Ward', 'Newtown Ward', 'Saint Davids Ward',
       'Heavitree Ward', 'Saint Leonards Ward', 'Priory Ward',
       'Saint Thomas Ward', 'Alphington Ward', 'Topsham Ward',
       'Mid Devon', 'North Devon', 'Torr

### Finally converting all to EPSG 27700

In [41]:
import os
import geopandas as gpd
for filename in os.listdir(r"osm_data"):
    if filename.endswith(".gpkg"):
        gdf = gpd.read_file(os.path.join(r"osm_data", filename))
        gdf = gdf.to_crs(epsg=27700)
        gdf = gdf[gdf.is_valid]
        gdf.to_file(os.path.join(r"osm_data", filename), driver="GPKG")

In [1]:
import geopandas as gpd
gdf_admin = gpd.read_file(r"osm_data\devon_boundaries.gpkg")
gdf_admin.name.value_counts()

name
Cornwall                4
Devon                   4
Dorset                  4
Somerset                3
Bickleigh               2
                       ..
Jurassic Coast          1
Devon and Torbay        1
Dorset and Wiltshire    1
Devon and Somerset      1
Torbay                  1
Name: count, Length: 545, dtype: int64

In [2]:
print(gdf_admin.name.value_counts())

name
Cornwall                4
Devon                   4
Dorset                  4
Somerset                3
Bickleigh               2
                       ..
Jurassic Coast          1
Devon and Torbay        1
Dorset and Wiltshire    1
Devon and Somerset      1
Torbay                  1
Name: count, Length: 545, dtype: int64


In [3]:
gdf_admin[gdf_admin.name == "Dorset"]

,visible,tags,lon,timestamp,lat,version,id,changeset,name,operator,boundary,marker,start_date,osm_type,website,admin_level,geometry
111,None,"{""ISO3166-2"":""GB-DOR"",""council_name"":""Dorset C...",NaN,1749892511,NaN,329,810850,0.0,Dorset,None,historic,None,None,relation,https://www.dorsetforyou.gov.uk/,None,"MULTIPOLYGON (((333922.201 101629.801, 333919...."
169,None,"{""alt_name"":""Dorsetshire"",""name:be-tarask"":""\u...",NaN,1749892511,NaN,147,693525998,0.0,Dorset,None,ceremonial,None,None,relation,None,None,"MULTIPOLYGON (((333922.201 101629.801, 333919...."
633,None,"{""designation"":""historic_county"",""type"":""bound...",NaN,1749892511,NaN,53,8569525539,0.0,Dorset,None,traditional,None,None,relation,None,None,"POLYGON ((328125.581 109105.481, 328119.524 10..."
643,None,"{""ISO3166-2"":""GB-DOR"",""alt_name"":""Dorsetshire""...",NaN,1759788729,NaN,29,13798541683,0.0,Dorset,None,administrative,None,2019-04-01,relation,https://www.dorsetcouncil.gov.uk/,6,"MULTIPOLYGON (((333922.201 101629.801, 333919...."


In [8]:
import joblib
data = joblib.load(r"artifacts\waterway_river_clyst_exeter_search.pkl").data
data.columns

Index(['visible', 'tags', 'lon', 'timestamp', 'lat', 'version', 'id',
       'changeset', 'waterfall', 'waterway', 'osm_type', 'canal', 'dock',
       'fish_pass', 'geometry'],
      dtype='object')

In [9]:
data

,visible,tags,lon,timestamp,lat,version,id,changeset,waterfall,waterway,osm_type,canal,dock,fish_pass,geometry
0,None,"{""visible"":false,""name"":""River Exe""}",NaN,1656425570,NaN,1,1074231820,NaN,None,river,way,None,None,None,"MULTILINESTRING ((295514.668 88906.558, 295524..."
1,None,"{""visible"":false,""layer"":""-1"",""name"":""Alphin B...",NaN,1563904621,NaN,2,446099915,NaN,None,river,way,None,None,None,"LINESTRING (291772.047 90272.36, 291814.661 90..."
2,None,"{""visible"":false}",NaN,1743084822,NaN,1,1372057805,NaN,None,river,way,None,None,None,"LINESTRING (292423.054 91585.851, 292460.561 9..."
3,None,"{""visible"":false,""boat"":""no""}",NaN,1713880951,NaN,2,1189743364,NaN,None,river,way,None,None,None,"LINESTRING (293096.831 90901.961, 293064.951 9..."
4,None,"{""visible"":false}",NaN,1743084822,NaN,3,1372056449,NaN,None,river,way,None,None,None,"LINESTRING (292409.544 91659.118, 292404.661 9..."
5,None,"{""destination"":""English Channel"",""name"":""River...",NaN,1761413151,NaN,15,3311908105,0.0,None,river,relation,None,None,None,"MULTILINESTRING ((296930.789 89445.763, 296921..."
6,None,"{""visible"":false}",NaN,1743084822,NaN,1,1372057807,NaN,None,river,way,None,None,None,"LINESTRING (292331.852 91737.528, 292314.041 9..."
7,None,"{""visible"":false}",NaN,1656425570,NaN,2,526463321,NaN,None,river,way,None,None,None,"MULTILINESTRING ((292299.985 91773.831, 292294..."
8,None,"{""visible"":false,""boat"":""no"",""name"":""Exeter Fl...",NaN,1515775004,NaN,13,8025018,NaN,None,river,way,None,None,None,"MULTILINESTRING ((290936.949 93882.328, 290941..."
9,None,"{""destination"":""River Exe"",""name"":""River Creed...",NaN,1603576672,NaN,4,3341652729,0.0,None,river,relation,None,None,None,"MULTILINESTRING ((290728.658 95499.546, 290718..."


### DOCUMENTATION : How to use this tool

In [1]:
import pandas as pd
import json
from utils.initialize_os_agents import OSAgentsInitializer
from utils.keys import set_api_keys
from utils.tools import human_send_message
set_api_keys()
import shutil
import joblib
import os

config = None
with open(r"agent_frameworks\agent_config_with_human_confirmation_expensive.json","rb") as file:
    config = json.load(file)

# Now that things are initialised
agent_archiecture = OSAgentsInitializer(config,"log",diff_dir=None).initialize_all_agents()

human_send_message(message="Find all buildings in Exeter that are more than 10m in height. Return the final artifact with the data and a plot",target_agent=[agent_archiecture["host_agent"]])

Openai and ngd key set successfully
Openai and ngd key set successfully
Model chosen gpt-5
Model chosen gpt-5
Model chosen gpt-4o-mini
Model chosen gpt-5
Model chosen gpt-4.1
bbox is  None <class 'NoneType'> True <class 'bool'>
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-5
Model chosen gpt-5
Model chosen gpt-5
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-5
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-5
Model chosen gpt-5
Model chosen gpt-5
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-5
Model chosen gpt-5
Model chosen gpt-5
Model chosen gpt-5
Model chosen gpt-5
Model chosen gpt-5


In [3]:
import shutil
import os
shutil.rmtree("artifacts")
shutil.rmtree("message_store")
os.makedirs("artifacts", exist_ok=True)
os.makedirs("message_store", exist_ok=True)